In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ARTIFACTS_PUBLIC = PROJECT_ROOT / "artifacts" / "public"
DATA_PUBLIC = PROJECT_ROOT / "data" / "public"
MODELS_PUBLIC = PROJECT_ROOT / "models" / "public"

In [2]:
import pandas as pd
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.ml.product import recommend_supplier as rs

V2_PATH = PROJECT_ROOT / "data" / "public" / "dataset_modelo_proveedor_v2_candidates.csv"
MODEL_PATH = PROJECT_ROOT / "models" / "public" / "baseline" / "model.pkl"
META_PATH = PROJECT_ROOT / "models" / "public" / "baseline" / "metadata.json"
MODEL_LABEL = "public_baseline"

df = pd.read_csv(V2_PATH)

required = {"fecha_evento", "albaran_id", "event_id", "producto_canonico", "proveedor_candidato", "coste_min_dia_proveedor", "target_elegido"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Faltan columnas en V2: {sorted(missing)}")

for c in ["albaran_id", "event_id", "producto_canonico", "proveedor_candidato"]:
    df[c] = df[c].astype(str).str.strip()
df["fecha_evento"] = pd.to_datetime(df["fecha_evento"], errors="coerce").dt.date
df["target_elegido"] = pd.to_numeric(df["target_elegido"], errors="coerce").fillna(0).astype(int)
df["coste_min_dia_proveedor"] = pd.to_numeric(df["coste_min_dia_proveedor"], errors="coerce")
df["rank_coste_dia_producto"] = pd.to_numeric(df.get("rank_coste_dia_producto"), errors="coerce")

### Justificación de postinferencia a grano `fecha_evento + albaran_id`

- En histórico real (`target_elegido=1`), cuando coexisten PRODUCT_002 y PRODUCT_003 en el mismo `fecha_evento + albaran_id`, la elección es coherente al 100% (`same_provider_real = 1.0`).
- El modelo actual, al rankear por `event_id` de forma independiente, rompe esa coherencia en parte de los casos (`model_top1_same_provider_rate < 1.0`).
- Una política puramente independiente por evento (mínimo coste por evento) rompe aún más la coherencia.
- Por tanto, la decisión final debe incorporar una capa de postinferencia a grano `fecha_evento + albaran_id` para coordinar proveedores entre líneas/productos del mismo albarán.
- Alcance explícito: este notebook valida coherencia por albarán; queda pendiente medir `Top-2` y `overrides improved/harmed/neutral` con la regla ya aplicada en postinferencia.


## Conclusiones

Los Notebooks 08 y 09 se han encontrado patrones claros que se pueden aplicar en posinferencia y que podrían mejorar considerablemente las métricas del modelo baseline con reglas deterministas. Esta línea de investigación sugiere:

- que buscar estos patrones es prioritario sobre mejora o enriquecimiento de features del modelo que ya tiene una buena puntuación:
    - de la investigación/exploración se derivan las siguientes reglas candidatas:
        - Regla prioritaria candidata: coherencia PRODUCT_002/PRODUCT_003 por fecha_evento + albaran_id
            Condición: cuando en el mismo albarán coexisten PRODUCT_002 y PRODUCT_003.
            Idea de regla: la decisión final postinferencia debe coordinar ambos para no dejar proveedores distintos.
            Hipótesis que la sostiene: el administrativo decide a grano albarán, no evento aislado.
            Evidencia: pair_groups_PRODUCT_002_PRODUCT_003=1807, real_same_provider_rate=1.0, model_top1_same_provider_rate=0.907028, model_top1_inconsistencies=168, min_event_same_provider_rate=0.339236;
        - R_EXP_001 (exploratoria): PRODUCT_003 en TERMINAL_001 con SUPPLIER_009 cuando está en mínimo
            Condición de datos: producto_canonico == 'PRODUCT_003' and terminal_compra == 'TERMINAL_001'.
            Hipótesis: en PRODUCT_003 el comportamiento real es estable y muy alineado a mínimo con SUPPLIER_009.
            Evidencia: support_events=45, fechas [2028-05-02, 2028-05-03, 2028-05-06, 2028-05-07, 2028-05-08], median_delta_eur_m3=0.00;
        - R_EXP_002 (exploratoria): PRODUCT_002 con SUPPLIER_009 aunque no sea mínimo
            Condición de datos: producto_canonico == 'PRODUCT_002' and provider_selected == 'SUPPLIER_009' and selected_in_min_cost == False;
            Hipótesis: hay restricciones no observadas en features actuales (crédito/cupo/logística) que pesan más que el mínimo precio puro.
            Evidencia: support_events=6, fechas [2028-05-03, 2028-05-06, 2028-05-07, 2028-05-08], median_delta_eur_m3=114.84;
- que las reglas deterministas también podrían dar lugar a una optimización en etapas posteriores del proyecto.

### DETALLE

La regla candidata principal que encontramos hoy es esta: **si en un mismo `fecha_evento + albaran_id` aparecen PRODUCT_002 y PRODUCT_003, la decisión final debería coordinar ambos productos para evitar proveedores distintos dentro del mismo albarán**.

La hipótesis salió de una pregunta muy concreta:  
**“¿El administrativo decide realmente por evento (línea) o decide por albarán completo?”**  
Cuando lo medimos, vimos que en histórico real la coherencia es total (`real_same_provider_rate = 1.0`), pero el modelo por evento rompe esa coherencia en parte (`model_top1_same_provider_rate = 0.907028`, `168` inconsistencias en `1807` grupos PRODUCT_002/PRODUCT_003). Además, una política “mínimo por evento” rompe aún más esa coherencia (`0.339236`). Por eso la regla de postinferencia por albarán tiene sentido técnico y de negocio.

Regla exploratoria 1 (`R_EXP_001`): en PRODUCT_003 (`TERMINAL_001`), SUPPLIER_009 aparece como elección estable cuando está en mínimo.  
La hipótesis fue: **“en PRODUCT_003 la decisión parece mucho más pegada al mínimo real”**.  
Evidencia: `45` eventos, en varias fechas de mayo, con `median_delta_eur_m3 = 0.00`.

Regla exploratoria 2 (`R_EXP_002`): en PRODUCT_002 hay eventos donde se elige SUPPLIER_009 aunque no sea mínimo.  
La hipótesis fue: **“hay variables operativas no capturadas (crédito/cupo/logística) que pesan más que el precio mínimo puro”**.  
Evidencia: `6` eventos, `median_delta_eur_m3 = 114.84`.

Conclusión clara:  
- **Regla lista para validar en postinferencia**: coherencia PRODUCT_002/PRODUCT_003 por albarán.  
- **Reglas exploratorias**: PRODUCT_003 mínimo estable y PRODUCT_002 con sobrecoste (aún requieren backtesting extendido + validación con administrativo antes de activarlas).

Regla en claro: **si en un mismo `fecha_evento + albaran_id` aparecen PRODUCT_002 y PRODUCT_003, y PRODUCT_003 se adjudica a SUPPLIER_009, entonces PRODUCT_002 también debe adjudicarse a SUPPLIER_009**. En el histórico analizado no vimos ningún caso que rompa esa condición.